# AutoGen Basic Demo

## Package install
These are all the install commands. See also `requirements.txt`.
```bash
python3 -m venv .venv           # Create a Python virtual environment (MacOS/Linux)
python -m venv .venv            # Create a Python virtual environment (Windows)

source .venv/bin/activate       # Activate the virtual environment (MacOS/Linux)
.venv\Scripts\activate          # Activate the virtual environment (Windows)

pip install -r requirements.txt # Install packages from requirements.txt

## Add Jupyter Kernel
python -m ipykernel install --user --name=.venv --display-name "Python (.venv)" # Register .venv as a Jupyter kernel

```

In [1]:
from tavily import TavilyClient # For our search tool (Just an example tool)
from dotenv import load_dotenv # You should have your own .env file with required API keys
import os
from IPython.display import display, Markdown

# Load the .env file
load_dotenv(override=True)
# Make sure to define your own API keys in your .env file
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT= os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_VERSION = os.getenv("AZURE_OPENAI_VERSION")

In [2]:

# Print to verify loaded values
print("API Key:", os.getenv("AZURE_OPENAI_API_KEY"))
print("Endpoint:", os.getenv("AZURE_OPENAI_ENDPOINT"))
print("API Version:", os.getenv("AZURE_OPENAI_VERSION"))

API Key: 52220aca3ae5498fbe5ad757f139e6c4
Endpoint: https://ue2coai4l3aoa02.openai.azure.com/
API Version: 2025-01-01-preview


### Test the Azure OpenAI API

In [3]:
from openai import AzureOpenAI

client = AzureOpenAI(
  azure_endpoint = AZURE_OPENAI_ENDPOINT, 
  api_key=AZURE_OPENAI_API_KEY,  
  api_version=AZURE_OPENAI_VERSION
)

response = client.chat.completions.create(
    model="gpt-4o", # Available models include "gpt-4o", "gpt-4o-mini", and "text-embedding-3-small"
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Does Azure OpenAI support customer managed keys?"},
        {"role": "assistant", "content": "Yes, customer managed keys are supported by Azure OpenAI."},
        {"role": "user", "content": "Do other Azure AI services support this too? Be concise."}
    ]
)

print(response.choices[0].message.content)

Yes, many other Azure AI services, such as Azure Cognitive Services and Azure Machine Learning, also support customer-managed keys (CMKs) for enhanced data security and compliance.


## Tools
For this example, we'll be using [Tavily Search API](https://docs.tavily.com/docs/welcome), which provides an easy way to search the web, optimized for LLMs and RAG. Here's an example of how it works.

If interested in getting your own API key to test this code, just login and get one for free, [here](https://app.tavily.com/home).

In [4]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

response = tavily_client.qna_search("What is agentic AI?")

# Nicely prints out the final Markdown text
display(Markdown(response.replace("```", "")))

Agentic AI refers to systems or programs that autonomously perform tasks on behalf of users or other systems by designing their own workflows and utilizing available tools. These AI systems can execute complex tasks involving data-driven decision-making and action-taking with minimal human intervention. They can manage marketing campaigns, continuously monitor performance, adjust strategies, and optimize results based on feedback without constant human input. Agentic AI uses self-contained agents that autonomously complete tasks and manage workflows using machine learning, algorithms, and predictive analytics to make real-time decisions. These AI agents can analyze data, set goals, and take actions with decreasing human supervision, enabling businesses to automate specific tasks, enhance customer service, and drive strategic decision-making.

## Travel Planning using AutoGen
Let's dive into a simple example of travel planning using agents.

### Defining Agent Tools
Agent tools in AutoGen could be any python function. Let's define a function that uses the Tavily Search API we defined earlier (`tavily_client`).

In [5]:
# AutoGen Imports
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient # Azure OpenAI API
from autogen_agentchat.agents import AssistantAgent # Agent
from autogen_agentchat.conditions import TextMentionTermination # Termination condition
from autogen_agentchat.teams import RoundRobinGroupChat # Agent design pattern
from autogen_agentchat.ui import Console # To view agent outputs

def search_tool(query: str) -> str:
    answer = tavily_client.search(query=query)
    return answer

### Define the LLM Client
For this project, we will be using the Azure resource provided by the challenge organizers. Make sure to use `AzureOpenAIChatCompletionClient`. [Click here to learn more](https://microsoft.github.io/autogen/dev/reference/python/autogen_ext.models.openai.html#autogen_ext.models.openai.AzureOpenAIChatCompletionClient).

In [6]:
client = AzureOpenAIChatCompletionClient(
    model="gpt-4o",
    api_version=AZURE_OPENAI_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    temperature=0,
    model_capabilities={
        "vision": False,
        "function_calling": True,
        "json_output": False,
    },
)

### Define Agents
Here we define a variety of agents to help support our goal of travel planning:

1. **`Research Agent`:** Make sure travel plan research is complete and facilitate conversation
2. **`Accommodations Search Agent:`** Search the web for accommodations (using `search_tool`)
3. **`Attractions Search Agent:`** Search the web for attractions (using `search_tool`)
4. **`Restaurants Search Agent:`** Search the web for restaurants (using `search_tool`)
5. **`Travel Summary Agent:`** Take in suggestions from travel research done previously and make the complete plan

In [7]:
research_agent = AssistantAgent(
    name="research_agent",
    model_client=client,
    description="A helpful assistant that ensures proper research is done to complete a travel plan.",
    system_message=
        """
        You are a research agent. You should ensure that all the proper research is done to complete the travel plan. 
        1. Accommodations
        2. Attractions
        3. Restaurants
        Facilitate the other agents in their research and guide the conversation so that all the necessary information is gathered.
        """,
)

accommodations_search_agent = AssistantAgent(
    name="accommodations_search_agent",
    model_client=client,
    description="A helpful assistant that can search for accommodations.",
    system_message=
            """
            You are an accommodations_search_agent agent. You can search for information on any topic. ALWAYS call your tool based on the query given to you.

            Your only tool is search_tool - use it to search for accommodations.
            You make only one search call at a time.
            Once you have the results, confer back to the research_agent.

            search_tool[query]:
            Description: Search to internet to find accommodations in your desired city.
            Parameter: query - The query related to searching for accommodations.
            Example: search_tool["Places to stay in Rome"] would present search result for stays in Rome.
            """,
    tools=[search_tool],
    reflect_on_tool_use=True,
)

attractions_search_agent = AssistantAgent(
    name="attractions_search_agent",
    model_client=client,
    description="A helpful assistant that can search for attractions.",
    system_message=
            """
            You are an attractions_search_agent agent. You can search for information on any topic. ALWAYS call your tool based on the query given to you.

            Your only tool is search_tool - use it to search for attractions.
            You make only one search call at a time.
            Once you have the results, confer back to the research_agent.

            search_tool[query]:
            Description: Search to internet to find attractions in your desired city.
            Parameter: query - The query related to searching for attractions.
            Example: search_tool["Tourist attractions in Rome"] would present search result for tourist attractions in Rome.
            """,
    tools=[search_tool],
)

restaurants_search_agent = AssistantAgent(
    name="restaurants_search_agent",
    model_client=client,
    description="A helpful assistant that can search for restaurants.",
    system_message=
            """
            You are an restaurants_search_agent agent. You can search for information on any topic. 

            Your only tool is search_tool - use it to search for restaurants.
            You make only one search call at a time.
            Once you have the results, confer back to the research_agent.

            search_tool[query]:
            Description: Search to internet to find restaurants in your desired city.
            Parameter: query - The query related to searching for restaurants.
            Example: search_tool["Best restaurants in Rome"] would present search result for restaurants in Rome.
            """,
    tools=[search_tool],
    reflect_on_tool_use=True,
)

travel_summary_agent = AssistantAgent(
    name="travel_summary_agent",
    model_client=client,
    description="A helpful assistant that can summarize the travel plan.",
    system_message=
        """
        You are a helpful assistant that can take in all of the suggestions and advice from the other agents and provide a detailed final travel plan. 
        You must ensure that the final plan is integrated and complete. YOUR FINAL RESPONSE MUST BE THE COMPLETE PLAN. 
        When the plan is complete and all perspectives are integrated, you can respond with TERMINATE.
        """,
)

### Set Termination Condition
It's important to set a termination condition so that agents don't get stuck in infinite loops. For now, we'll just implement termination to indicate the end of the conversation, but you can find [other termination conditions here](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/termination.html).

In [8]:
termination = TextMentionTermination("TERMINATE")

### RoundRobinGroupChat
Round Robin is one of the group chat classes [predefined in AutoGen](https://microsoft.github.io/autogen/stable/reference/python/autogen_agentchat.teams.html#autogen_agentchat.teams.RoundRobinGroupChat). In this agent design pattern, participating agents all take turns speaking in round robin fashion.

In [9]:
group_chat = RoundRobinGroupChat(
    [research_agent, accommodations_search_agent, attractions_search_agent, restaurants_search_agent, travel_summary_agent], termination_condition=termination,
)
await Console(group_chat.run_stream(task=
                                    """
                                    Please help me plan a trip from San francisco to Lake Tahoe spanning 4days from March 16th to March 19th, 2025. 
                                    The travel should be planned for a single person with a budget of $1,000.
                                    """
                                    ))

---------- user ----------

                                    Please help me plan a trip from St. Petersburg to Rockford spanning 3 days from March 16th to March 18th, 2025. 
                                    The travel should be planned for a single person with a budget of $1,700.
                                    


/Users/tejajuttu/Library/CloudStorage/OneDrive-EY/Documents/Agentic AI/AI Agent Challenge/Experiments_TJ/.venv/lib/python3.11/site-packages/autogen_agentchat/agents/_assistant_agent.py:386: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect.
  model_result = await self._model_client.create(


---------- research_agent ----------
Great! Let's break this trip into manageable parts to ensure we stay within your $1,700 budget and cover all aspects of your travel. Here's how we'll proceed:

### 1. **Transportation**
   - Flights from St. Petersburg, FL (PIE or TPA) to Rockford, IL (RFD or nearby airports like Chicago O'Hare).
   - Local transportation in Rockford (rental car, rideshare, or public transport).

### 2. **Accommodations**
   - Budget-friendly hotels, motels, or Airbnb options in Rockford for 3 nights (March 16–18, 2025).

### 3. **Attractions**
   - Key attractions in Rockford to explore during your stay.

### 4. **Restaurants**
   - Affordable and highly-rated dining options for breakfast, lunch, and dinner.

### 5. **Miscellaneous**
   - Any additional costs like parking, entrance fees, or souvenirs.

Let’s start with **transportation** first. I’ll research flight options and local transportation. Other agents can begin looking into accommodations, attractions, an

/Users/tejajuttu/Library/CloudStorage/OneDrive-EY/Documents/Agentic AI/AI Agent Challenge/Experiments_TJ/.venv/lib/python3.11/site-packages/autogen_agentchat/agents/_assistant_agent.py:468: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect.
  model_result = await self._model_client.create(llm_messages, cancellation_token=cancellation_token)


---------- accommodations_search_agent ----------
### Accommodations in Rockford, IL (March 16–18, 2025)
Here are some options for your stay in Rockford:

1. **Embassy Suites by Hilton Rockford Riverfront**
   - Highly rated with modern amenities.
   - Close to attractions like Anderson Japanese Gardens.
   - [More details on Tripadvisor](https://www.tripadvisor.co.uk/Hotels-g36619-Rockford_Illinois-Hotels.html)

2. **Hampton Inn Rockford**
   - Includes breakfast and is budget-friendly.
   - [More details on Tripadvisor](https://www.tripadvisor.com/LastMinute-g36619-Rockford_Illinois-Hotels.html)

3. **Holiday Inn Rockford, an IHG Hotel**
   - Features a fitness center, restaurant, and bar.
   - [More details on Booking.com](https://www.booking.com/accommodation/city/us/rockford.html)

4. **Staybridge Suites Rockford, an IHG Hotel**
   - Offers barbecue facilities and free WiFi.
   - [More details on Booking.com](https://www.booking.com/accommodation/city/us/rockford.html)

5. **OYO H

TaskResult(messages=[TextMessage(source='user', models_usage=None, content='\n                                    Please help me plan a trip from St. Petersburg to Rockford spanning 3 days from March 16th to March 18th, 2025. \n                                    The travel should be planned for a single person with a budget of $1,700.\n                                    ', type='TextMessage'), TextMessage(source='research_agent', models_usage=RequestUsage(prompt_tokens=135, completion_tokens=248), content="Great! Let's break this trip into manageable parts to ensure we stay within your $1,700 budget and cover all aspects of your travel. Here's how we'll proceed:\n\n### 1. **Transportation**\n   - Flights from St. Petersburg, FL (PIE or TPA) to Rockford, IL (RFD or nearby airports like Chicago O'Hare).\n   - Local transportation in Rockford (rental car, rideshare, or public transport).\n\n### 2. **Accommodations**\n   - Budget-friendly hotels, motels, or Airbnb options in Rockford for

In [10]:
group_chat = RoundRobinGroupChat(
    [research_agent, accommodations_search_agent, attractions_search_agent, restaurants_search_agent, travel_summary_agent], termination_condition=termination,
)
await Console(group_chat.run_stream(task=
                                    """
                                    Please help me plan a trip from San francisco to Lake Tahoe spanning 4days from March 16th to March 19th, 2025. 
                                    The travel should be planned for a single person with a budget of $1,000.
                                    """
                                    ))

---------- user ----------

                                    Please help me plan a trip from San francisco to Lake Tahoe spanning 4days from March 16th to March 19th, 2025. 
                                    The travel should be planned for a single person with a budget of $1,000.
                                    
---------- research_agent ----------
Got it! Let’s plan your 4-day trip from San Francisco to Lake Tahoe from March 16th to March 19th, 2025, while staying within your $1,000 budget. Here's how we'll organize the planning:

---

### **1. Transportation**
   - **To Lake Tahoe**: Options include driving (rental car or personal vehicle), bus, or train.
   - **Local Transportation**: Options for getting around Lake Tahoe (rental car, public transport, or rideshare).

---

### **2. Accommodations**
   - Budget-friendly hotels, motels, or Airbnb options in Lake Tahoe for 3 nights (March 16–18, 2025).

---

### **3. Attractions**
   - Activities and must-see spots in Lake Ta

TaskResult(messages=[TextMessage(source='user', models_usage=None, content='\n                                    Please help me plan a trip from San francisco to Lake Tahoe spanning 4days from March 16th to March 19th, 2025. \n                                    The travel should be planned for a single person with a budget of $1,000.\n                                    ', type='TextMessage'), TextMessage(source='research_agent', models_usage=RequestUsage(prompt_tokens=450, completion_tokens=279), content="Got it! Let’s plan your 4-day trip from San Francisco to Lake Tahoe from March 16th to March 19th, 2025, while staying within your $1,000 budget. Here's how we'll organize the planning:\n\n---\n\n### **1. Transportation**\n   - **To Lake Tahoe**: Options include driving (rental car or personal vehicle), bus, or train.\n   - **Local Transportation**: Options for getting around Lake Tahoe (rental car, public transport, or rideshare).\n\n---\n\n### **2. Accommodations**\n   - Budget-f

# FINAL Travel Plan after "Thinking" for several minutes

Here is the final output from the `travel_summary_agent`

---

---------- travel_summary_agent ----------
### Final Integrated Travel Plan: San Francisco to Lake Tahoe (March 16–19, 2025)

#### **Day 1: March 16, 2025 (Sunday)**
- **Morning**:
  - Depart from San Francisco by rental car for a scenic 3.5–4-hour drive to Lake Tahoe. Estimated cost: ~$50/day for rental + ~$50 for gas.
  - Alternatively, take a bus (~$95 round trip) or Amtrak train + bus (~$27–$110).

- **Afternoon**:
  - Check-in at **Big Pines Mountain House of Tahoe** in South Lake Tahoe. (~$100/night)
  - Relax and explore the nearby Lakeside Beach.

- **Evening**:
  - Dinner at **Riva Grill**, offering lakefront dining with seafood and steak. Estimated cost: $30–$40.
  - Return to the hotel and rest for the next day.

---

#### **Day 2: March 17, 2025 (Monday)**
- **Morning**:
  - Breakfast at **The Getaway Cafe**, known for its hearty breakfast. Estimated cost: $15–$20.
  - Visit **Emerald Bay State Park** for stunning views and hiking trails. Free entry.

- **Afternoon**:
  - Lunch at **Maggie's**, a cozy spot with great views. Estimated cost: $20–$25.
  - Take a ride on **The Gondola at Heavenly** for panoramic views of the lake. Ticket cost: ~$60.

- **Evening**:
  - Dinner at **Cafe Fiore**, an award-winning Italian fine dining restaurant. Estimated cost: $40–$50.
  - Optional: Explore the nightlife or relax at the hotel.

---

#### **Day 3: March 18, 2025 (Tuesday)**
- **Morning**:
  - Breakfast at the hotel or a local cafe.
  - Spend the morning skiing or snowboarding at **Heavenly Ski Resort** or **Northstar California Resort**. Ski pass: ~$150 (rentals extra).

- **Afternoon**:
  - Lunch at **Evan's American Gourmet Cafe**, offering upscale dining. Estimated cost: $30–$40.
  - Visit **Tallac Historic Site** to explore historic estates and gardens. Free entry.

- **Evening**:
  - Dinner at **The Idle Hour**, a lakeside spot with great ambiance. Estimated cost: $30–$40.
  - Return to the hotel and relax.

---

#### **Day 4: March 19, 2025 (Wednesday)**
- **Morning**:
  - Breakfast at the hotel or a local cafe.
  - Check out and drive back to San Francisco. (~3.5–4 hours)
  - Alternatively, take the bus or train back.

---

#### **Budget Breakdown**
1. **Transportation**: $150–$250 (rental car + gas or bus/train)
2. **Accommodations**: $300 (3 nights at Big Pines Mountain House of Tahoe)
3. **Dining**: $160–$240 (average $50/day)
4. **Attractions & Activities**: $210 (Gondola ride, ski pass, etc.)
5. **Total**: $820–$1,000

---

#### **Notes**
- This plan stays within your $1,000 budget if you choose mid-range options for dining and activities.
- All costs are estimates and may vary slightly based on availability and booking time.

Would you like assistance with bookings or any further customization? If everything looks good, you're all set for a fantastic trip! **TERMINATE**


# Final Travel Plan
Here is the final output from the `travel_summary_agent`

---
### Complete Travel Plan from St. Petersburg to Rockford (March 16th to March 18th, 2025)

#### Budget Overview
- **Total Budget:** $1,700
- **Accommodations:** $300
- **Attractions:** $200
- **Restaurants:** $150
- **Transportation:** $1,050

### Day 1: March 16th, 2025

#### Transportation
- **Flight from St. Petersburg to Rockford:**
  - **Estimated Cost:** $600 round trip
  - **Airlines:** Check for flights with major airlines like Delta, American Airlines, or United
  - **Local Transportation:** Rental car for 3 days ($150) or rideshare services ($100)

#### Accommodations
- **Hotel:** Fairfield Inn & Suites by Marriott Rockford
  - **Price:** $100 per night
  - **Amenities:** Free Wi-Fi, breakfast included, fitness center, indoor pool
  - **Location:** Conveniently located near major attractions

#### Attractions
- **Anderson Japanese Gardens**
  - **Description:** Beautifully landscaped Japanese gardens with ponds, waterfalls, and traditional structures.
  - **Admission:** $10

#### Restaurants
- **Dinner:** Prairie Street Brewing Co.
  - **Cuisine:** American, Brewery
  - **Price Range:** $15-$30 per meal

### Day 2: March 17th, 2025

#### Attractions
- **Morning:** Burpee Museum of Natural History
  - **Description:** Museum featuring dinosaur skeletons, fossils, and exhibits on natural history.
  - **Admission:** $12

- **Afternoon:** Discovery Center Museum
  - **Description:** Interactive science museum with hands-on exhibits for all ages.
  - **Admission:** $10

- **Evening:** Coronado Performing Arts Center
  - **Description:** A historic theater offering a variety of performances and events.
  - **Admission:** Varies by event

#### Restaurants
- **Breakfast:** The Norwegian
  - **Cuisine:** Scandinavian, Breakfast & Brunch
  - **Price Range:** $10-$25 per meal

- **Lunch:** GreenFire Restaurant Bar & Bakery
  - **Cuisine:** American, Bakery
  - **Price Range:** $15-$35 per meal

- **Dinner:** Franchesco's Ristorante
  - **Cuisine:** Italian
  - **Price Range:** $20-$40 per meal

### Day 3: March 18th, 2025

#### Attractions
- **Morning:** Tinker Swiss Cottage Museum & Gardens
  - **Description:** Historic house museum with beautiful gardens and guided tours.
  - **Admission:** $8

- **Afternoon:** Rock Cut State Park
  - **Description:** A large state park offering hiking, fishing, and picnicking opportunities.
  - **Admission:** Free

- **Evening:** Rockford Art Museum
  - **Description:** Featuring contemporary and regional art, this museum hosts rotating exhibits and educational programs.
  - **Admission:** $8

#### Restaurants
- **Breakfast:** Lino's
  - **Cuisine:** Italian
  - **Price Range:** $15-$30 per meal

- **Lunch:** Prairie Street Brewing Co.
  - **Cuisine:** American, Brewery
  - **Price Range:** $15-$30 per meal

- **Dinner:** GreenFire Restaurant Bar & Bakery
  - **Cuisine:** American, Bakery
  - **Price Range:** $15-$35 per meal

### Summary of Costs
- **Accommodations:** $300 (3 nights at $100 per night)
- **Attractions:** $48 (Anderson Japanese Gardens, Burpee Museum, Discovery Center, Tinker Swiss Cottage, Rockford Art Museum)
- **Restaurants:** $150 (Estimated $50 per day)
- **Transportation:** $750 (Flight $600 + Local Transportation $150)

### Total Estimated Cost: $1,248

This plan leaves you with a buffer of $452 for any additional expenses or adjustments. Enjoy your trip to Rockford!